# 1. Cấu hình GitHub & Mount Drive

In [ ]:
import os
from google.colab import drive

# --- THAY ĐỔI THÔNG TIN TẠI ĐÂY ---
GITHUB_TOKEN = ""  # Dán Personal Access Token (PAT) của bạn vào đây
GITHUB_USER = "nck345"
GITHUB_REPO = "PBL5_test"
BRANCH = "main"

# Đường dẫn thư mục chứa dataset trên Drive (Nơi bạn đã tải lên folder 'dataset')
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/Colab Notebooks/PBL5_test"
# --------------------------------

drive.mount('/content/drive')

Mounted at /content/drive


# 2. Tải Code từ GitHub về Colab

In [ ]:
%cd /content

repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

if not os.path.exists(GITHUB_REPO):
    print(f"🚀 Đang clone repository {GITHUB_REPO}...")
    !git clone {repo_url}
else:
    print(f"🔄 Cập nhật code mới nhất từ GitHub...")
    %cd {GITHUB_REPO}
    !git pull origin {BRANCH}

%cd /content/{GITHUB_REPO}

/content
🚀 Đang clone repository PBL5_test...
Cloning into 'PBL5_test'...
remote: Enumerating objects: 2936, done.
remote: Counting objects: 100% (341/341), done.
remote: Compressing objects: 100% (202/202), done.
remote: Total 2936 (delta 201), reused 247 (delta 113), pack-reused 2595 (from 2)
Receiving objects: 100% (2936/2936), 144.73 MiB | 19.69 MiB/s, done.
Resolving deltas: 100% (206/206), done.
/content/PBL5_test


# 3. Kết nối Dataset và thư mục kết quả từ Drive

In [ ]:
# Danh sách các thư mục cần đồng bộ với Drive
folders_to_link = ['dataset', 'models', 'logs']

for folder in folders_to_link:
    drive_folder = os.path.join(DRIVE_PROJECT_PATH, folder)
    local_folder = os.path.join(f"/content/{GITHUB_REPO}", folder)

    # Tạo thư mục trên Drive nếu chưa có (để lưu models/logs)
    if not os.path.exists(drive_folder):
        os.makedirs(drive_folder, exist_ok=True)
        print(f"📁 Đã tạo thư mục {folder} trên Drive")

    # Xóa thư mục/link cũ tại local (nếu có) để tạo link mới chính xác
    if os.path.exists(local_folder):
        if os.path.islink(local_folder):
            os.unlink(local_folder)
        else:
            import shutil
            shutil.rmtree(local_folder)

    # Tạo liên kết (Symbolic Link)
    !ln -s "{drive_folder}" "{local_folder}"
    print(f"🔗 Đã liên kết: {local_folder} -> {drive_folder}")

🔗 Đã liên kết: /content/PBL5_test/dataset -> /content/drive/MyDrive/Colab Notebooks/PBL5_test/dataset
🔗 Đã liên kết: /content/PBL5_test/models -> /content/drive/MyDrive/Colab Notebooks/PBL5_test/models
🔗 Đã liên kết: /content/PBL5_test/logs -> /content/drive/MyDrive/Colab Notebooks/PBL5_test/logs


# 4. Cài đặt thư viện và Chạy Training

In [ ]:
import torch

print("📦 Đang cài đặt thư viện...")
!pip install -r requirements.txt

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    
    print("🔥 Bắt đầu quá trình huấn luyện mô hình cơ sở từ Scratch (trên các bộ dữ liệu công khai)...")
    !python train_all.py
    
    print("\n🔥 Bắt đầu quá trình Fine-tuning mô hình trên dữ liệu tự thu ESP32...")
    !python train_all.py --fine_tuning
else:
    print("❌ LỖI: Không tìm thấy GPU! Hãy kiểm tra cài đặt Runtime.")

📦 Đang cài đặt thư viện...
✅ GPU: Tesla T4
🔥 Bắt đầu quá trình huấn luyện...
Loading configuration...
Device: cuda
Seed: 42

Preparing data for all models...
🚀 BƯỚC 1: Tải và Nạp Động dữ liệu (Dynamic Loading)...
Nạp dữ liệu từ các thư mục: 100% 4/4 [00:57<00:00, 14.43s/it]

📊 Tổng hợp dữ liệu (Combined):
  Train: X=(70110, 128, 9), y=(70110,)
  Val:   X=(15023, 128, 9), y=(15023,)
  Test:  X=(15026, 128, 9), y=(15026,)

⚙️ Đang fit StandardScaler trên toàn bộ tập Train kết hợp...
✅ Đã lưu Global Scaler tại: models/final_model/scaler_global.pkl
Benchmark threshold mode: fixed (0.50)

STARTING TRAINING SEQUENCE: ['lstm', 'stacked_lstm', 'ensemble']


TRAINING MODEL: LSTM
  Total parameters: 19,265

Starting training on cuda
Epochs: 50 | LR: 0.001 | Early Stopping: True

Epoch [  1/50] | Train Loss: 0.0544 Acc: 66.31% | Val Loss: 0.0505 Acc: 69.68% | LR: 0.001000 | Time: 6.8s
  -> Saved best checkpoint (val_loss: 0.0505)
Epoch [  2/50] | Train Loss: 0.0495 Acc: 68.94% | Val Loss: 0.0519 